In [64]:
import json
import os
import urllib
from pathlib import Path
from IPython.display import clear_output

In [3]:
stops = set()
for file in os.listdir("geojson_cache"):
    with open(os.path.join("geojson_cache", file)) as f:
        data = json.load(f)
        for feature in data["features"]:
            if "properties" in feature and "codparag" in feature["properties"]:
                stops.add(feature["properties"]["codparag"])

In [89]:
if Path("trips_ids.txt").exists():
    with open("trips_ids.txt", "r") as f:
        trips = set(line.strip() for line in f)
else:
    trips = set()

print(f"Pre loaded {len(trips)} trips")

date = "2026-05-19"
s = 40
with open("trips_ids.txt", "a") as f:
    for i, id in enumerate(list(stops)[s:]):
        encoded_id = urllib.parse.quote(id, safe='')
        url = f"https://paragens.amp.pt/acarto2/get_horarios_prg?dia={date}&id={encoded_id}"
        
        request = urllib.request.Request(url)
        with urllib.request.urlopen(request) as response:
            data = json.loads(json.loads(response.read()))
            for horario in data["horarios"]:
                if horario["trip_id"] not in trips:
                    f.write(horario["trip_id"] + "\n")
                trips.add(horario["trip_id"])
        
        if i % 20 == 0:
            clear_output()
            display(f"{i/(len(stops)-s)*100:.2f}%")

'99.87%'